# SPAI Project 1 - Group 4

## `l-diversity` implementation on `Top Down Greedy`for dataset: `CAHOUSING`

First result of baseline model, without any changes made:

Top-down Greedy, cahousing, k=2

* NCP: 0.156
* CAVG after: 1.204
* DM after: 55316
* Runtime: 12.603 s

For comparability of the experiment results, we added `random_seed(42)`. This is the 'official' baseline result for comparison:

- k = 2
- NCP = 0.155
- CAVG = 1.208
- DM = 56764
- Partitions = 9029

### First _l-diversity_ extension

The original TDG algorithm was extended with a distinct _l_-diverstiy check.

For a partition to be considered valid, it must satisfy:
* k-anonymity: partition size >= k
* distinct _l-diversity_: at least _l_ distinct values of the sensitive attribute

For the `cahousing` dataset, `ocean_proximity` was selected as the sensitive attribute.
The first implementation rejected a proposed TDG split whenever one of the resulting child partitions violated l-diversity.

#### Experiment

**Configuration:**

- Dataset: `cahousing`
- Algorithm: Top-down Greedy
- k = 2
- _l_ = 2
- Sensitive attribute: `ocean_proximity`
- Random seed: 42

**Results:**

- NCP: 1.000
- CAVG*: 10320.000
- DM: 426009600
- Runtime: 1.766 s
- Final number of partitions: 1

*The dataset contains 20640 records. With k=2 the final result consists of exactly one equivalence class, thus CAVG becomes:
$$\frac{20640}{1 \cdot 2} = 10320$$

### Observation

The algorithm retained the entire dataset as one equivalence class.
The inital split produced at least one child partition that did not satisfy _l-diversity_. Since the implementation immediately rejected such a split, the parent partition was retained instead.
Although the final partition satisfies 2-diversity, this leads to complete generalization of the QI and therefore very poor data utility.
This indicates that simply rejecting the first invalid TDG split is too restrictive. A better strategy should attempt alternative splits before giving up on partitioning.


**Strategy:**

Replace the current "reject the first invalid split" behavior with a helper that tries several candidate TDG splits and accepts the first one that satisfies k-anonymity and l-diversity.

### After fixing-l-diversity

**Configuration:**

- Dataset: `cahousing`
- Algorithm: Top-down Greedy
- k = 2
- _l_ = 2
- Sensitive attribute: `ocean_proximity`
- Random seed: 42

**Results:**

- NCP: 0.469
- CAVG: 15.426
- DM: 4360456
- Runtime: 17.171 s
- Final number of partitions: 743

In [2]:
import pandas as pd

path = "results/cahousing/topdown/cahousing_anonymized_k2_l2.csv"

df_anon = pd.read_csv(path, sep=";")

qi_columns = [
    "longitude",
    "latitude",
    "housing_median_age",
    "median_income",
    "median_house_value"
]

sensitive_column = "ocean_proximity"

equivalence_classes = df_anon.groupby(qi_columns, dropna=False)

class_sizes= equivalence_classes.size()

diversity = equivalence_classes[sensitive_column].nunique()

print("Number of equivalence classes:", len(class_sizes))
print("Minimum equivalence class size:", class_sizes.min())
print("Minimum l-diversity:", diversity.min())

print("Violations of k=2:", (class_sizes < 2).sum())
print("Violations of l=2:", (diversity < 2).sum())

Number of equivalence classes: 669
Minimum equivalence class size: 2
Minimum l-diversity: 2
Violations of k=2: 0
Violations of l=2: 0


## Preliminary Results table

| k | l | Method | NCP | CAVG | DM | ECs | k violations | l violations |
|---:|---:|---|---:|---:|---:|---:|---:|---:|
| 2 | – | TDG k-anonymity | 0.155 | 1.208 | 56,764 | TBD | 0 | – |
| 2 | 2 | TDG + l-diversity | 0.469 | 15.426 | 4,360,456 | 669 | 0 | 0 |

## Classification Utility Evaluation

Classification performance is used to evaluate the utility of the
anonymized data.

The target variable is `ocean_proximity`.

A Random Forest classifier is trained and evaluated on:

1. the original dataset,
2. the k-anonymous TDG dataset,
3. the k-anonymous + l-diverse TDG dataset.

The predefined train/validation split supplied with the `cahousing`
dataset is used for all experiments.

Performance is measured using weighted F1 score.

In [2]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [14]:
TARGET_COLUMN = "ocean_proximity"
ID_COLUMN = "ID"

QI_COLUMNS = [
    "longitude",
    "latitude",
    "housing_median_age",
    "median_income",
    "median_house_value"
]

QI_MIDPOINTS = {
    col: (
        pd.to_numeric(original_df[col]).min()
        + pd.to_numeric(original_df[col]).max()
    ) / 2
    for col in QI_COLUMNS
}

In [4]:
original_df = pd.read_csv(
    "data/cahousing/cahousing.csv",
    sep=";"
)

original_df.head()

,ID,longitude,latitude,housing_median_age,mean_rooms,mean_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,15574,-116.78,33.0,7,7.354154,1.146730,5102,1697,5.5102,221000,1
1,15154,-117.03,33.0,6,7.972727,1.029870,2693,770,7.7569,387400,1
2,15207,-117.07,33.0,4,5.593190,1.127240,2211,1116,4.2500,281600,1
3,15208,-117.07,33.0,4,5.390459,1.098940,3775,1698,4.9550,241500,1
4,15215,-117.10,33.0,5,6.574215,1.108142,7417,2358,5.9094,261100,1


In [5]:
with open("data/cahousing/cahousing_train.txt", "r") as f:
    train_indexes = [int(i) for i in f.read().splitlines()]

with open("data/cahousing/cahousing_val.txt", "r") as f:
    val_indexes = [int(i) for i in f.read().splitlines()]

print("Training records:", len(train_indexes))
print("Validation records:", len(val_indexes))

Training records: 16512
Validation records: 4128


#### Helper for TDG generalized values

TDG can produce QI values such as `4.2,5.7`. For classification we need umerical classification and therefore this helper function to replace each generalized interval by its midpoint:

$$
x=\frac{lower+upper}{2}
$$


* Exact values are converted directly to float.
* Generalized intervals are represented by their midpoint.
* Full generalization (*) is converted to the midpoint of the original attribute domain

In [15]:
def generalized_to_numeric(value, column):
    value = str(value).strip()

    # Fully generalized attribute
    if value == "*":
        return QI_MIDPOINTS[column]

    # TDG numeric range, e.g. "4.2,5.7"
    if "," in value:
        low, high = value.split(",", 1)
        return (float(low) + float(high)) / 2

    # Alternative hierarchy range representation
    if "~" in value:
        low, high = value.split("~", 1)
        return (float(low) + float(high)) / 2

    # Original / non-generalized value
    return float(value)

#### Feature-preparation function

* Original dataset: the QIs are converted to normal floats.
* Anonymized dataset: generalized intervals become their midpoint.

In [17]:
def prepare_features(df):
    feature_columns = [
        col for col in df.columns
        if col not in [ID_COLUMN, TARGET_COLUMN]
    ]

    X = df[feature_columns].copy()

    for col in feature_columns:
        if col in QI_COLUMNS:
            X[col] = X[col].apply(
                lambda value: generalized_to_numeric(value, col)
            )
        else:
            X[col] = pd.to_numeric(X[col])

    return X

#### Align anonymized results by ID

Restore the original row order using the ID column.

The anonymized output is sorted by ID before being written to storage.

For classification comparison we need to restore the order.

In [8]:
def align_to_original_order(original_df, anon_df):
    if not original_df[ID_COLUMN].is_unique:
        raise ValueError("Original IDs are not unique")

    if not anon_df[ID_COLUMN].is_unique:
        raise ValueError("Anonymized IDs are not unique")

    aligned = (
        original_df[[ID_COLUMN]]
        .merge(
            anon_df,
            on=ID_COLUMN,
            how="left",
            validate="one_to_one"
        )
    )

    if aligned.isnull().any().any():
        raise ValueError("Some records could not be aligned by ID")

    return aligned

#### Classification Comparison Experiment

In [9]:
k2_df = pd.read_csv(
    "results/cahousing/topdown/cahousing_anonymized_k2.csv",
    sep=";"
)

k2_l2_df = pd.read_csv(
    "results/cahousing/topdown/cahousing_anonymized_k2_l2.csv",
    sep=";"
)

In [ ]:
# Aligning the datasets

k2_df = align_to_original_order(
    original_df,
    k2_df
)

k2_l2_df = align_to_original_order(
    original_df,
    k2_l2_df
)

In [11]:
# Sanity check
print(original_df[ID_COLUMN].head().tolist())
print(k2_df[ID_COLUMN].head().tolist())
print(k2_l2_df[ID_COLUMN].head().tolist())

[15574, 15154, 15207, 15208, 15215]
[15574, 15154, 15207, 15208, 15215]
[15574, 15154, 15207, 15208, 15215]


In [18]:
# Preparing features and target

X_original = prepare_features(original_df)
X_k2 = prepare_features(k2_df)
X_k2_l2 = prepare_features(k2_l2_df)

y = original_df[TARGET_COLUMN]

In [19]:
# Sanity checks
print(X_original.shape)
print(X_k2.shape)
print(X_k2_l2.shape)

print("\nMissing values:")
print(X_original.isna().sum().sum())
print(X_k2.isna().sum().sum())
print(X_k2_l2.isna().sum().sum())

print("\nData types:")
print(X_k2.dtypes)

(20640, 9)
(20640, 9)
(20640, 9)

Missing values:
0
0
0

Data types:
longitude             float64
latitude              float64
housing_median_age    float64
mean_rooms            float64
mean_bedrooms         float64
population              int64
households              int64
median_income         float64
median_house_value    float64
dtype: object


In [20]:
# Random Forest evaluation function
def evaluate_random_forest(X, y, train_indexes, val_indexes):
    model = RandomForestClassifier(
        n_estimators=300,
        oob_score=True,
        min_samples_split=5,
        max_depth=10,
        random_state=42
    )

    X_train = X.iloc[train_indexes]
    y_train = y.iloc[train_indexes]

    X_val = X.iloc[val_indexes]
    y_val = y.iloc[val_indexes]

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    return f1_score(
        y_val,
        predictions,
        average="weighted"
    )

In [21]:
# Evaluating all three datasets
f1_original = evaluate_random_forest(
    X_original,
    y,
    train_indexes,
    val_indexes
)

f1_k2 = evaluate_random_forest(
    X_k2,
    y,
    train_indexes,
    val_indexes
)

f1_k2_l2 = evaluate_random_forest(
    X_k2_l2,
    y,
    train_indexes,
    val_indexes
)

print("Original data:", f1_original)
print("TDG k=2:", f1_k2)
print("TDG k=2, l=2:", f1_k2_l2)

Original data: 0.9538000467408884
TDG k=2: 0.8979454988946695
TDG k=2, l=2: 0.8349625809349434


In [22]:
# Classification table
classification_df = pd.DataFrame([
    {
        "method": "Original data",
        "k": None,
        "l": None,
        "weighted_f1": f1_original
    },
    {
        "method": "TDG k-anonymity",
        "k": 2,
        "l": None,
        "weighted_f1": f1_k2
    },
    {
        "method": "TDG + l-diversity",
        "k": 2,
        "l": 2,
        "weighted_f1": f1_k2_l2
    }
])

classification_df

,method,k,l,weighted_f1
0,Original data,NaN,NaN,0.953800
1,TDG k-anonymity,2.0,NaN,0.897945
2,TDG + l-diversity,2.0,2.0,0.834963


## Results table pt. 2
| k | l | Method | NCP | CAVG | DM | ECs | k violations | l violations | Weighted F1 |
|---:|---:|---|---:|---:|---:|---:|---:|---:|---:|
| 2 | – | TDG k-anonymity | 0.155 | 1.208 | 56,764 | TBD | 0 | – | 0.897945|
| 2 | 2 | TDG + l-diversity | 0.469 | 15.426 | 4,360,456 | 669 | 0 | 0 |0.834963|